# meeting_extracted_w1_t1_mail — W1 第一支：T1-a 取數 → T1-m 初洗 → T1-c 參與者

- 用途：把 `b_internal_meeting_record` 清成「一列一封信的可讀版」與「參與者 token 表」，供 T1-b（分段、token 化）與展示系統讀。全是規則，無 LLM。規格 §4.0、§4.0b、§4.1b；偏離處見 `meeting_extracted_migration/STATUS.md` D10–D14。
- 輸入 table：`{catalog}.{schema}.b_internal_meeting_record`（不讀 `mail_body_html`）、`s_semantic_entity_alias` / `s_semantic_entity`（網域 → 公司 / 發言方）
- 輸出 table：`s_meeting_record_mail_{unit}`（MERGE on `pk_seq`）、`s_meeting_participants_{unit}`（本批 pk 先 DELETE 再 append）、`s_meeting_record_skipped_{unit}`（D34 隔離表：不符規則的信，MERGE on `pk_seq`、不蓋 `note`）
- 參數（widgets）：`catalog`、`schema`、`source_table`、`run_mode`（incremental / backfill）、`date_from` / `date_to`（backfill 與首跑用，對 `mail_date`）、`unit`、`cleaner_version`、`sample_limit`、`reprocess_pks`（逗號分隔 pk，強制納入本批）、`dry_run`
- 排程：W1 每週一；首次先 `run_mode = backfill`、`dry_run = true` 看關卡 → `false`。serverless 可跑（不 cache）。
- 負責人 / 更新日期：（填）/ 2026-10-15

| cell | 做什麼 |
|---|---|
| [c03] | 初洗：HTML entity、圖片佔位、base64 圖片、連結縮文字、Word / CSS 殘渣、免責段；來源選擇；`body_conflict`；`body_hash` |
| [c04] | `mail_from` 四段、主旨去 FW / RE、**標頭區塊偵測**（md `**From:**` 與純文字 `From:` 兩種）、`When:` / `Sent:` 日期、整列組裝 |
| [c05] | 參與者：收件清單切分、顯示名拆中英、群組判定、**token 配發**（同 pk 同人同 token）、IMCEAEX 假位址排除 |
| [c06] | Spark 接線：struct UDF、網域後綴比對別名表、`dup_of_pk`、隔離規則 `SKIP_RULES` / `split_skipped` |
| [c12] | 關卡只印不停（D34）：0 封、隔離封數與原因、`body_conflict` > 10% 印 WARN；不符規則的信在 [c11] 隔離，其餘照寫 |

隱私：本 notebook 的兩張輸出表是 S3（含人名、email），權限同 Bronze。dry_run 的樣本輸出刻意不含姓名、email、正文。

In [ ]:
# [c01] params
# W1 第一支：T1-a 取數 → T1-m 初洗落 mail 表 → T1-c 參與者落 participants 表。無 LLM。
# 由 job parameters 覆蓋；widget 建立過後改預設值不會生效，要重設先 dbutils.widgets.removeAll()。
from datetime import date, datetime

dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.text("source_table", "b_internal_meeting_record")
dbutils.widgets.dropdown("run_mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("date_from", "2026-06-01")   # backfill：對 mail_date，含；incremental 首跑（表空）也用這個窗
dbutils.widgets.text("date_to", "2026-12-31")
dbutils.widgets.text("unit", "bu")                 # D4：整表都算 BU；只影響 unit 欄的值
dbutils.widgets.text("cleaner_version", "0.2")     # 改 [c03] / [c04] 規則就升；舊版本的列會在 incremental 被重洗。0.2：D28 來源選擇改看初洗後長度
dbutils.widgets.text("sample_limit", "0")          # 開發用：> 0 只取最新 N 封；正式跑 0
dbutils.widgets.text("reprocess_pks", "")          # 逗號分隔 pk_seq：不管水位強制納入本批；隔離表的信處理好後重跑用（D34）
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"])  # true：算完印關卡與樣本，不寫表


def _date(s: str) -> date:
    return datetime.strptime(s.strip(), "%Y-%m-%d").date()


cfg = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "source_table": dbutils.widgets.get("source_table").strip(),
    "run_mode": dbutils.widgets.get("run_mode").strip(),
    "date_from": _date(dbutils.widgets.get("date_from")),
    "date_to": _date(dbutils.widgets.get("date_to")),
    "unit": dbutils.widgets.get("unit").strip(),
    "cleaner_version": dbutils.widgets.get("cleaner_version").strip(),
    "sample_limit": int(dbutils.widgets.get("sample_limit")),
    "reprocess_pks": [int(s) for s in dbutils.widgets.get("reprocess_pks").split(",") if s.strip()],
    "dry_run": dbutils.widgets.get("dry_run") == "true",
}
assert cfg["catalog"] and cfg["schema"] and cfg["source_table"], "catalog / schema / source_table 不可為空"
assert cfg["date_from"] <= cfg["date_to"], "date_from 不可晚於 date_to"
assert cfg["unit"] and cfg["cleaner_version"], "unit / cleaner_version 不可為空"
print(cfg)

In [ ]:
# [c02] imports
# [c02]～[c05] 是純 Python（不碰 spark / dbutils），tests/ 直接載入；[c06] 是 Spark 接線。
import hashlib
import html
import json
import re
import unicodedata
from datetime import datetime

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StructType

BODY_FIELDS = ["mail_body_md", "mail_without_fw", "mail_body"]  # 文字欄，依序嘗試
HTML_FIELD = "mail_body_html"  # D29：三個文字欄初洗後都 < 150 字才讀（ETL 對約 30% 的信文字欄只剩 Word 殘渣）
SOURCE_COLS = [
    "pk_seq", "drupal_id", "drupal_link", "create_date", "mail_date", "mail_datetime",
    "mail_from", "mail_to", "mail_cc", "mail_bcc", "mail_subject",
    *BODY_FIELDS, HTML_FIELD, "is_fw", "has_attach", "is_test",
]
MIN_BODY_CHARS = 150      # md 採用門檻（§4.1 (a) 1）；也是 mail 層去重門檻（太短的信不算重複）
OVERSIZE_CHARS = 200_000  # D12：初洗後仍 > 200k 才算 oversize（規格是初洗前；base64 圖片刪掉後多半正常）
RESOLVER_VERSION = "0.1"  # T1-c 規則版本；改 [c05] 規則就升

HEADER_KEYS = ("From", "Sent", "To", "Cc", "Bcc", "Subject", "When", "Where")
# 標頭標記兩種寫法：md `**From:**`（可在行中）、純文字行首 `From:`。group(1) 或 group(2) 是鍵名。
KEY_RE = re.compile(
    r"\*\*(From|Sent|To|Cc|Bcc|Subject|When|Where):\*\*[ \t]*"
    r"|^[ \t]*(From|Sent|To|Cc|Bcc|Subject|When|Where):[ \t]*",
    re.M,
)
LINE_END_KEYS = {"From", "Sent", "Subject", "Where"}  # 區塊最後一個鍵若是這些，值只到行尾
ORIG_SEP_RE = re.compile(r"-{3,}\s*Original (?:Message|Appointment)\s*-{3,}")
MAIL_FROM_RE = re.compile(r"^(\d{6,8})/([^/]+)/([^/]+)/([A-Z0-9]+)$")
SUBJECT_PREFIX_RE = re.compile(r"^(\s*(fwd|fw|re|回覆|轉寄)\s*([:：]|\s)\s*)+", re.I)
EMAIL_RE = re.compile(r"^[A-Za-z0-9._%+\-]+@([A-Za-z0-9\-]+(?:\.[A-Za-z0-9\-]+)*\.[A-Za-z]{2,})$")
RECIPIENT_RE = re.compile(r"^(.*?)\s*<([^<>]+)>\s*$")
CJK_RE = re.compile(r"[一-鿿]+")
GROUP_HINT_RE = re.compile(r"\bPM\b|\bDL\b|^HQ\.|\bAll\b|\bTeam\b|\bGroup\b|\bMembers?\b", re.I)

# 初洗（§4.1 (b) 規則 1–3 ＋ Phase 0 加的 base64 / 連結 / CSS 殘渣）
MD_IMG_RE = re.compile(r"!\[[^\]]*\]\(cid:[^)]*\)")
B64_RE = re.compile(r"(?:!\[[^\]]*\]\()?data:image/[A-Za-z0-9.+\-]+;base64,[A-Za-z0-9+/=]+\)?")
MD_LINK_RE = re.compile(r"\[([^\]]+)\]\((?:https?://|mailto:)[^)]*\)")
MD_ESCAPE_RE = re.compile(r"\\([_*#])")
WORD_JUNK_RE = re.compile(
    r"^[ \t]*(?:Normal|DocumentEmail|MicrosoftInternetExplorer4|false|true|EN-US|ZH-TW|X-NONE|\d{1,3})[ \t]*$\n?",
    re.M,
)
CSS_JUNK_RE = re.compile(  # 0.2：整行刪只限 400 字內的行；殘渣行都很短，正文行若含 mso- 不能整行陪葬
    r"^(?=[^\n]{0,400}$).*(?:mso-|font-family:|@font-face|LatentStyle|panose-1|<!--\[if|<!\[endif\]|UnhideWhenUsed=).*$\n?"
    r"|^[ \t]*(?:<!--|-->|/\*[^\n]*\*/)[ \t]*$\n?",  # 0.2：Word 樣式區塊的 <!-- / --> / /* Font Definitions */ 獨立行
    re.M,
)
DISCLAIMER_RE = re.compile(
    r"CONFIDENTIAL|保密(?:資訊|信息|訊息|文件|內容)|This email contains information|如您並非|has pledged to source"
    r"|sent from outside of the company|intended recipient|All rights including intellectual property",
    re.I,
)
DASH_LINE_RE = re.compile(r"^\s*[-_=*]{10,}\s*$")
# HTML → 文字（D29，只用於 mail_body_html 備援）。Outlook / Word HTML：<head> 裡是 xml 條件註解與 <style>，
# 正文在 <div class=WordSection1>，段落 <p class=MsoNormal>、空行是 <p><o:p>&nbsp;</o:p></p>、標頭用 <b>From:</b> … <br>。
# 沒關閉的條件註解 / 註解 / <head> <style>（ETL 截斷的 HTML 常停在 <style><!-- 中間）一律吃到字串結尾，
# 否則幾萬字的 CSS 會變成一行「正文」（2026-10-19 第三次 dry_run：42 封被這樣誤救、dup_of_pk 跳到 11%）。
HTML_DROP_RE = re.compile(
    r"<!--\[if(?:.*?<!\[endif\]-->|.*$)"
    r"|<!--(?:.*?-->|.*$)"
    r"|<(head|style|script|title|xml)\b(?:.*?</\1\s*>|.*$)",
    re.S | re.I,
)
HTML_IMG_RE = re.compile(r"<img\b[^>]*>", re.I)
HTML_TR_RE = re.compile(r"<tr\b[^>]*>", re.I)
HTML_CELL_END_RE = re.compile(r"</t[dh]\s*>", re.I)
HTML_BLOCK_BEFORE_CELL_END_RE = re.compile(r"(?:</p\s*>|</div\s*>|<br\s*/?>)(?=\s*</t[dh]\s*>)", re.I)
HTML_BLOCK_END_RE = re.compile(r"</(?:p|div|tr|li|h[1-6]|table|ul|ol|blockquote)\s*>|<br\s*/?>|<hr\s*/?>", re.I)
HTML_TAG_RE = re.compile(r"<[^>]+>")
# body_conflict 只用來偵測「三個正文欄講不同會議」，不是正式客戶對照（那走 s_semantic_entity_alias）
CUSTOMER_PATTERNS = {
    "acer": re.compile(r"\bacer\b|宏碁", re.I), "asus": re.compile(r"\basus\b|華碩", re.I),
    "dell": re.compile(r"\bdell\b", re.I), "hp": re.compile(r"\bHP\b"),
    "lenovo": re.compile(r"\blenovo\b|聯想", re.I), "msi": re.compile(r"\bMSI\b|微星"),
    "tcl": re.compile(r"\bTCL\b"), "hisense": re.compile(r"\bhisense\b|海信", re.I),
    "samsung": re.compile(r"\bsamsung\b|\bVD\b|三星", re.I), "tpv": re.compile(r"\bTPV\b|冠捷"),
    "sony": re.compile(r"\bsony\b", re.I), "lg": re.compile(r"\bLGE?\b"),
    "benq": re.compile(r"\bbenq\b|明基", re.I), "qisda": re.compile(r"\bqisda\b|佳世達", re.I),
}
TOKEN_RE = re.compile(r"\w+")  # 粗 token（中文無空白會成長串，Jaccard 偏低，所以還要配客戶集合）

# 日期解析（Sent: / When:）
MONTHS = {m: i for i, m in enumerate(
    ["january", "february", "march", "april", "may", "june", "july", "august", "september",
     "october", "november", "december"], start=1)}
EN_DT_RE = re.compile(
    r"(?:[A-Za-z]+,\s*)?([A-Za-z]+)\s+(\d{1,2}),\s*(\d{4})(?:\s+(\d{1,2}):(\d{2})(?::(\d{2}))?\s*([AP]M)?)?", re.I
)
ZH_DT_RE = re.compile(r"(\d{4})年(\d{1,2})月(\d{1,2})日(?:[^\d]{0,12}?(上午|下午)?\s*(\d{1,2}):(\d{2}))?")

In [ ]:
# [c03] pure_text_clean
# T1-m 初洗：只做「哪個正文可信、去雜訊」。保留標頭、人名、email、表格；不 token 化、不簡轉繁（那是 T1-b）。
def norm_space(s: str | None) -> str:
    """所有空白（含 U+00A0、換行）壓成一格並去頭尾。"""
    return re.sub(r"\s+", " ", (s or "").replace("\xa0", " ")).strip()


def unescape_text(s: str | None) -> str:
    """HTML entity 還原（&lt; &nbsp; &amp;…）、U+00A0 → 空白、換行統一。"""
    t = html.unescape(s or "")
    return t.replace("\xa0", " ").replace("\r\n", "\n").replace("\r", "\n")


def drop_disclaimer_paragraphs(t: str) -> str:
    """規則 3：免責 / ESG / 外部信警語**從命中的那一行起到段落結尾**刪掉，命中行之前的行保留（0.2；0.1 是整段刪，
    HTML 轉出來的文字段落間常沒有空行，整段刪會連坐正文）；純分隔線段落也刪。段落 = 空白行分隔。"""
    kept = []
    for para in re.split(r"\n[ \t]*\n", t):
        lines = para.splitlines()
        hit = next((i for i, ln in enumerate(lines) if DISCLAIMER_RE.search(ln)), None)
        if hit is not None:
            lines = lines[:hit]
        if all(DASH_LINE_RE.match(ln) or not ln.strip() for ln in lines):
            continue
        kept.append("\n".join(lines))
    return "\n\n".join(kept)


def html_to_text(raw: str | None) -> str:
    """D29：Outlook / Word HTML → 純文字，給 clean_display 再初洗。只用 stdlib regex（cluster 不裝套件）。
    原始碼裡的換行是軟換行（先換成空白），再用區塊標籤產生換行：</p> </div> <br> </tr> → 換行、
    </td> → ` | `（<tr> 起手 `| `，變成 md 表格列，T1-b 的表格規則接得上）；<img data:…> → [image]、cid 圖刪；
    <a> 只留文字；其餘標籤刪；最後還原 entity。輸出不保證有空白行（Outlook 的空行是空段落，會留下來）。"""
    if not raw:
        return ""
    t = raw.replace("\r\n", "\n").replace("\r", "\n")
    t = HTML_DROP_RE.sub(" ", t)
    t = re.sub(r"[ \t]*\n[ \t]*", " ", t)
    t = HTML_IMG_RE.sub(lambda m: "[image]" if "data:image" in m.group(0) else "", t)
    t = HTML_BLOCK_BEFORE_CELL_END_RE.sub("", t)
    t = HTML_TR_RE.sub("\n| ", t)
    t = HTML_CELL_END_RE.sub(" | ", t)
    t = HTML_BLOCK_END_RE.sub("\n", t)
    t = HTML_TAG_RE.sub("", t)
    t = html.unescape(t).replace("\xa0", " ")
    lines = [re.sub(r"[ \t]{2,}", " ", ln).strip() for ln in t.split("\n")]
    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()


def clean_display(raw: str | None, source_field: str) -> str:
    """規則 1–3 ＋ Phase 0 加的規則，三個來源欄都套（D10）。輸出 markdown，可直接渲染。"""
    t = unescape_text(raw)
    t = MD_IMG_RE.sub("", t)              # 1. 圖片佔位 ![…](cid:…) / ![…](data:…)
    t = B64_RE.sub("[image]", t)          # 1b. 表格儲存格內 base64 圖片（每轉寄翻倍的元兇）
    t = MD_LINK_RE.sub(r"\1", t)          # 1c. [文字](網址) → 文字
    t = MD_ESCAPE_RE.sub(r"\1", t)        # md 跳脫 \_ \* \# 還原
    t = CSS_JUNK_RE.sub("", t)            # 2. Word / Outlook CSS 殘渣行
    t = WORD_JUNK_RE.sub("", t)           # 2. 獨立殘渣行（Normal / EN-US / ≤3 位純數字…）
    t = drop_disclaimer_paragraphs(t)     # 3. 免責與 ESG 聲明
    t = re.sub(r"[ \t]+\n", "\n", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()


def choose_body(src: dict) -> tuple[str, str | None, str]:
    """§4.1 (a) 1–3 ＋ D28 / D29：依 md → mail_without_fw → mail_body 的順序，取**第一個初洗後 ≥ 150 字**的
    文字欄；三個都不到才把 mail_body_html 轉文字、初洗，≥ 150 字就用它（body_source_field = mail_body_html）；
    四個都不到就取初洗後最長的非空者；全空 → none。回傳 (body_source_field, raw, display)。
    背景：ETL 對約 30% 的信（146 封）三個文字欄都只剩 Word 文件屬性殘渣（<!--[if gte mso 9]> 裡的 XML 被當文字、
    到 <style> 就截斷），正文只在 mail_body_html（中位數 34k 字）。0.1「md 不到 150 字就用非空的 mail_without_fw」
    把殘渣選走、初洗剩 27 字，這些信整封變 noise。"""
    cands = {f: clean_display(src.get(f), f) for f in BODY_FIELDS}
    for f in BODY_FIELDS:
        if len(cands[f]) >= MIN_BODY_CHARS:
            return f, src.get(f), cands[f]
    cands[HTML_FIELD] = clean_display(html_to_text(src.get(HTML_FIELD)), HTML_FIELD)
    if len(cands[HTML_FIELD]) >= MIN_BODY_CHARS:
        return HTML_FIELD, src.get(HTML_FIELD), cands[HTML_FIELD]
    f = max([*BODY_FIELDS, HTML_FIELD], key=lambda k: len(cands[k]))  # 平手取順序在前者
    if not cands[f]:
        return "none", None, ""
    return f, src.get(f), cands[f]


def tokens(s: str) -> set[str]:
    return {w.lower() for w in TOKEN_RE.findall(s or "") if len(w) >= 2}


def jaccard(a: set, b: set) -> float | None:
    if not a or not b:
        return None
    return len(a & b) / len(a | b)


def customers_in(s: str) -> set[str]:
    return {k for k, p in CUSTOMER_PATTERNS.items() if p.search(s or "")}


def body_conflict(src: dict, chosen_field: str, chosen_display: str) -> bool:
    """§4.1 (a) 4：選用來源與任一其他非空來源 Jaccard < 0.3 且客戶集合都非空又不相交 → 衝突。不擋萃取。"""
    if chosen_field == "none":
        return False
    ta, ca = tokens(chosen_display), customers_in(chosen_display)
    for f in BODY_FIELDS:  # HTML 只是文字欄全空時的備援，不拿來比
        if f == chosen_field:
            continue
        other = clean_display(src.get(f), f)
        if not other:
            continue
        j = jaccard(ta, tokens(other))
        cb = customers_in(other)
        if j is not None and j < 0.3 and ca and cb and not (ca & cb):
            return True
    return False


def normalize_text(s: str | None) -> str:
    """去重 / 穩定鍵共用的正規化：NFKC（全半形統一）、去標點、去空白、去控制字元、小寫。
    T1-b 的 content_hash 與 T2 的 item_key 都用這個。"""
    t = unicodedata.normalize("NFKC", s or "")
    return "".join(ch for ch in t if unicodedata.category(ch)[0] not in "PZC").lower()


def sha1_hex(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8")).hexdigest()


def body_hash(display: str) -> str | None:
    """mail 層去重鍵；太短的信（noise）不參與去重 → NULL。"""
    if len(display or "") < MIN_BODY_CHARS:
        return None
    return sha1_hex(normalize_text(display))

In [ ]:
# [c04] pure_mail_fields
# mail 層欄位：mail_from 四段、主旨、標頭區塊（T1-c 與 T1-b 共用）、When / Sent 日期、整列組裝。
def parse_mail_from(s: str | None) -> dict:
    """`工號/中文姓名/帳號/部門代碼` → 四欄。不符格式：name_zh 放原字串，其餘 NULL，ok = False。"""
    raw = norm_space(s)
    m = MAIL_FROM_RE.match(raw)
    if not m:
        return {"emp_id": None, "name_zh": raw or None, "account": None, "dept_code": None, "ok": False}
    return {"emp_id": m.group(1), "name_zh": m.group(2).strip(), "account": m.group(3).strip(),
            "dept_code": m.group(4), "ok": True}


def subject_clean(s: str | None) -> str | None:
    t = norm_space(s)
    t = SUBJECT_PREFIX_RE.sub("", t).strip()
    return t or None


def _last_value_end(text: str, pos: int, limit: int, key: str) -> int:
    """區塊最後一個鍵的值到哪裡：From / Sent / Subject / Where 到行尾；To / Cc / Bcc / When 到第一個空白行。"""
    if key in LINE_END_KEYS:
        nl = text.find("\n", pos)
        return limit if nl == -1 else min(nl, limit)
    m = re.compile(r"\n[ \t]*\n").search(text, pos, limit)
    return m.start() if m else limit


def find_header_blocks(text: str) -> list[dict]:
    """找出正文裡的轉寄 / 回覆標頭區塊，依文件順序（上 → 下）。
    區塊 = 一個 From 標記起、到下一個 From 標記前的 Sent / To / Cc / Bcc / Subject / When / Where；
    只有 From 沒有 Sent 也沒有 Subject 的不算（避免正文裡的「From:」）。
    回傳 [{"start", "end", "fields": {key: 值（空白已壓）}}]；start / end 是在 text 的字元偏移。"""
    marks = [(m.start(), m.end(), m.group(1) or m.group(2)) for m in KEY_RE.finditer(text)]
    blocks, i = [], 0
    while i < len(marks):
        if marks[i][2] != "From":
            i += 1
            continue
        j = i + 1
        while j < len(marks) and marks[j][2] != "From":
            j += 1
        group = marks[i:j]
        limit = marks[j][0] if j < len(marks) else len(text)
        fields, end = {}, group[0][1]
        for k, (_, me, key) in enumerate(group):
            ve = group[k + 1][0] if k + 1 < len(group) else _last_value_end(text, me, limit, key)
            fields.setdefault(key, norm_space(text[me:ve]))
            end = ve
        if "Sent" in fields or "Subject" in fields:
            blocks.append({"start": group[0][0], "end": end, "fields": fields})
        i = j
    return blocks


def find_key_value(text: str, key: str) -> str | None:
    """正文裡第一個 `key:` 的值（不限在標頭區塊內；When / Where 用）。"""
    marks = [(m.start(), m.end(), m.group(1) or m.group(2)) for m in KEY_RE.finditer(text)]
    for k, (_, me, name) in enumerate(marks):
        if name != key:
            continue
        limit = marks[k + 1][0] if k + 1 < len(marks) else len(text)
        return norm_space(text[me:_last_value_end(text, me, limit, key)]) or None
    return None


def parse_datetime(s: str | None) -> datetime | None:
    """Sent: / When: 的日期時間。英文長格式（星期可有可無、秒可有可無、AM/PM）與中文 `yyyy年m月d日 上午/下午 h:mm`。
    只取第一個時間（When 的區間取開始）。解析不到 → None。"""
    t = norm_space(s)
    if not t:
        return None
    m = EN_DT_RE.search(t)
    if m and m.group(1).lower() in MONTHS:
        mon, day, year = MONTHS[m.group(1).lower()], int(m.group(2)), int(m.group(3))
        hh, mm = int(m.group(4) or 0), int(m.group(5) or 0)
        ap = (m.group(7) or "").upper()
        if ap == "PM" and hh < 12:
            hh += 12
        if ap == "AM" and hh == 12:
            hh = 0
        try:
            return datetime(year, mon, day, hh, mm)
        except ValueError:
            return None
    m = ZH_DT_RE.search(t)
    if m:
        year, mon, day = int(m.group(1)), int(m.group(2)), int(m.group(3))
        hh, mm = int(m.group(5) or 0), int(m.group(6) or 0)
        if m.group(4) == "下午" and hh < 12:
            hh += 12
        try:
            return datetime(year, mon, day, hh, mm)
        except ValueError:
            return None
    return None


def build_mail_row(src: dict, unit: str, cleaner_version: str) -> dict:
    """Bronze 一列 → s_meeting_record_mail_bu 一列（dup_of_pk / cleaned_at 由 Spark 端填）。"""
    field, raw, display = choose_body(src)
    blocks = find_header_blocks(display)
    fw = parse_mail_from(src.get("mail_from"))
    when_raw = find_key_value(display, "When")
    where_raw = find_key_value(display, "Where")
    return {
        "pk_seq": src["pk_seq"],
        "drupal_id": src.get("drupal_id"),
        "drupal_link": src.get("drupal_link"),
        "unit": unit,
        "create_date": src.get("create_date"),
        "mail_date": src.get("mail_date"),
        "mail_datetime": src.get("mail_datetime"),
        "subject_raw": src.get("mail_subject"),
        "subject_clean": subject_clean(src.get("mail_subject")),
        "forwarder_emp_id": fw["emp_id"],
        "forwarder_name_zh": fw["name_zh"],
        "forwarder_account": fw["account"],
        "forwarder_dept_code": fw["dept_code"],
        "forwarder_dept_name": None,                       # 待決 #13
        "forwarder_token": "P01" if fw["name_zh"] else None,  # T1-c 一律把轉寄人配 P01
        "mail_to_raw": src.get("mail_to") or None,
        "mail_cc_raw": src.get("mail_cc") or None,
        "mail_bcc_raw": src.get("mail_bcc") or None,
        "meeting_when": parse_datetime(when_raw),
        "meeting_where": where_raw,
        "body_source_field": field,
        "has_headers": bool(blocks),
        "body_conflict": body_conflict(src, field, display),
        "body_display": display or None,
        "raw_len": len(raw or ""),
        "display_len": len(display),
        "is_fw": src.get("is_fw"),
        "attach_count": int(src.get("has_attach") or 0),
        "oversize": len(display) > OVERSIZE_CHARS,
        "body_hash": body_hash(display),
        "dup_of_pk": None,
        "cleaner_version": cleaner_version,
        "cleaned_at": None,
    }

In [ ]:
# [c05] pure_participants
# T1-c：從 mail_from、頂層收件欄、正文標頭區塊抓人；同一 pk 內同一人共用一個 token。side / company 由 [c06] 查別名表補。
def split_recipients(value: str | None) -> list[tuple[str, str | None]]:
    """`顯示名 <email>; 顯示名; email` → [(顯示名, email 或 None)]。
    Exchange 的 IMCEAEX-… 假位址與不合法 email → None（顯示名保留）。"""
    out = []
    for item in re.split(r";", norm_space(value)):
        item = item.strip().strip("'\"")
        if not item:
            continue
        m = RECIPIENT_RE.match(item)
        if m:
            disp, addr = m.group(1).strip().strip("'\""), m.group(2).strip()
        elif EMAIL_RE.match(item):
            disp, addr = "", item
        else:
            disp, addr = item, None
        email = addr.lower() if addr and EMAIL_RE.match(addr) and not addr.upper().startswith("IMCEAEX") else None
        if not disp and email:
            disp = email.split("@")[0]
        out.append((disp, email))
    return out


def parse_display_name(disp: str) -> tuple[str | None, str | None, str]:
    """顯示名 → (name_zh, name_en, kind)。
    含中文 → 中文連續字串是 name_zh，其餘是 name_en（AUO 慣例「英名 中文名」）；
    `姓, 名` → name_en = `名 姓`；全大寫無小寫、或含 PM / Team / HQ. 等 → group。"""
    s = norm_space(disp)
    if not s:
        return None, None, "person"
    zh = "".join(CJK_RE.findall(s)) or None
    en = norm_space(CJK_RE.sub(" ", s)).strip(" ,") or None
    if en and "," in en:
        last, first = (p.strip() for p in en.split(",", 1))
        en = f"{first} {last}".strip()
    is_group = False
    if not zh and en:
        letters = [c for c in en if c.isalpha()]
        if letters and all(c.isupper() for c in letters):
            is_group = True
        if GROUP_HINT_RE.search(en):
            is_group = True
    return zh, en, "group" if is_group else "person"


def identity_keys(e: dict) -> list[str]:
    """同一人的辨識鍵：email 優先，其次中文名，其次英文名（正規化）。"""
    keys = []
    if e.get("email"):
        keys.append("e:" + e["email"])
    if e.get("name_zh"):
        keys.append("z:" + normalize_text(e["name_zh"]))
    elif e.get("name_en"):
        keys.append("n:" + normalize_text(e["name_en"]))
    return keys


def assign_tokens(entries: list[dict]) -> None:
    """依出現順序配 P01、P02…（群組 G01…）；任一辨識鍵命中既有者共用 token。原地寫入 e['token']。"""
    by_key, counters = {}, {"P": 0, "G": 0}
    for e in entries:
        keys = identity_keys(e)
        tok = next((by_key[k] for k in keys if k in by_key), None)
        if tok is None:
            prefix = "G" if e["kind"] == "group" else "P"
            counters[prefix] += 1
            tok = f"{prefix}{counters[prefix]:02d}"
        for k in keys:
            by_key.setdefault(k, tok)
        e["token"] = tok


def make_entry(disp: str, email: str | None, role: str, segment_no: int) -> dict:
    zh, en, kind = parse_display_name(disp)
    return {
        "segment_no": segment_no, "role": role, "kind": kind,
        "display_name_raw": norm_space(disp) or None, "name_zh": zh, "name_en": en,
        "email": email, "email_domain": email.split("@")[1] if email else None,
        "emp_id": None, "account": None, "dept_code": None,
    }


def extract_participants(mail: dict) -> list[dict]:
    """build_mail_row 的輸出 → s_meeting_participants_bu 的列（side / company / resolved_at 先 NULL）。
    順序：轉寄人（segment 0）→ 頂層 to / cc / bcc（segment 0）→ 正文標頭區塊 1..n 的 From / To / Cc / Bcc。
    segment_no 與 T1-b 的段序對齊：第 n 個標頭區塊 = 第 n 段（文件順序，上 → 下）。"""
    entries = []
    if mail.get("forwarder_name_zh"):
        parts = [mail.get("forwarder_emp_id"), mail.get("forwarder_name_zh"),
                 mail.get("forwarder_account"), mail.get("forwarder_dept_code")]
        raw = "/".join(p for p in parts if p) if mail.get("forwarder_emp_id") else mail["forwarder_name_zh"]
        zh, en, _ = parse_display_name(mail["forwarder_name_zh"])
        entries.append({
            "segment_no": 0, "role": "forwarder", "kind": "person",
            "display_name_raw": raw, "name_zh": zh, "name_en": en, "email": None, "email_domain": None,
            "emp_id": mail.get("forwarder_emp_id"), "account": mail.get("forwarder_account"),
            "dept_code": mail.get("forwarder_dept_code"),
        })
    for role, col in (("to", "mail_to_raw"), ("cc", "mail_cc_raw"), ("bcc", "mail_bcc_raw")):
        for disp, email in split_recipients(mail.get(col)):
            entries.append(make_entry(disp, email, role, 0))
    for n, blk in enumerate(find_header_blocks(mail.get("body_display") or ""), start=1):
        for key, role in (("From", "from"), ("To", "to"), ("Cc", "cc"), ("Bcc", "bcc")):
            if key in blk["fields"]:
                for disp, email in split_recipients(blk["fields"][key]):
                    entries.append(make_entry(disp, email, role, n))
    assign_tokens(entries)
    seen, rows = set(), []
    for e in entries:
        key = (e["segment_no"], e["token"], e["role"])
        if key in seen:
            continue
        seen.add(key)
        rows.append({
            "pk_seq": mail["pk_seq"], **{k: e[k] for k in (
                "segment_no", "token", "role", "kind", "display_name_raw", "name_zh", "name_en",
                "email", "email_domain")},
            "side": None, "company": None,
            **{k: e[k] for k in ("emp_id", "account", "dept_code")},
            "dept_name": None, "title_hint": None,
            "resolver_version": RESOLVER_VERSION, "resolved_at": None,
        })
    return rows

In [ ]:
# [c06] spark_transforms
# Spark 接線。[c03]～[c05] 是逐封信的字串處理（標頭區塊、token 配發），用 Python UDF 回傳 struct 是最直接的寫法；
# 日量百封級、單封最大 2MB，UDF 成本可忽略。struct 的欄位順序 = 目標表 schema（由 [c11] 傳入），不另外手寫。
def _row_to_struct(row_dict: dict, schema: StructType) -> tuple:
    return tuple(row_dict.get(f.name) for f in schema.fields)


def transform_mail(df_src: DataFrame, mail_schema: StructType, unit: str, cleaner_version: str) -> DataFrame:
    """Bronze → mail 表形狀（dup_of_pk / cleaned_at 仍 NULL）。"""
    def _one(r):
        return _row_to_struct(build_mail_row(r.asDict(), unit, cleaner_version), mail_schema)

    udf = F.udf(_one, mail_schema)
    return df_src.select(udf(F.struct(*SOURCE_COLS)).alias("m")).select("m.*")


def transform_participants(df_mail: DataFrame, part_schema: StructType) -> DataFrame:
    """mail 表形狀 → 參與者列（side / company / resolved_at 仍 NULL）。"""
    def _many(r):
        return [_row_to_struct(p, part_schema) for p in extract_participants(r.asDict())]

    udf = F.udf(_many, ArrayType(part_schema))
    cols = ["pk_seq", "forwarder_emp_id", "forwarder_name_zh", "forwarder_account", "forwarder_dept_code",
            "mail_to_raw", "mail_cc_raw", "mail_bcc_raw", "body_display"]
    return df_mail.select(F.explode(udf(F.struct(*cols))).alias("p")).select("p.*")


def resolve_side(df_part: DataFrame, df_alias: DataFrame, df_entity: DataFrame) -> DataFrame:
    """email_domain 後綴比對 s_semantic_entity_alias（alias_type = email_domain）→ side / company。
    多個別名命中（auo.com 與 corpnet.auo.com）取最長者。轉寄人一律 auo / auo（§4.1b 規則 1）。
    entity.side = ignore（webex、gmail 等）→ side unknown、company NULL。"""
    al = (
        df_alias.where((F.col("alias_type") == "email_domain") & F.col("is_active"))
        .select(F.col("alias_norm").alias("dom_alias"), "entity_id")
        .join(df_entity.where("is_active").select("entity_id", F.col("side").alias("ent_side")), "entity_id", "left")
    )
    cond = (F.col("email_domain") == F.col("dom_alias")) | F.col("email_domain").endswith(
        F.concat(F.lit("."), F.col("dom_alias"))
    )
    j = df_part.join(F.broadcast(al), cond, "left")
    w = Window.partitionBy("pk_seq", "segment_no", "token", "role").orderBy(F.length("dom_alias").desc_nulls_last())
    j = j.withColumn("_rn", F.row_number().over(w)).where("_rn = 1")
    side = (
        F.when(F.col("role") == "forwarder", F.lit("auo"))
        .when(F.col("ent_side").isin("auo", "customer", "third_party"), F.col("ent_side"))
        .otherwise(F.lit("unknown"))
    )
    company = (
        F.when(F.col("role") == "forwarder", F.lit("auo"))
        .when(F.col("ent_side") == "ignore", F.lit(None))
        .otherwise(F.col("entity_id"))
    )
    return j.withColumns({"side": side, "company": company}).drop("dom_alias", "entity_id", "ent_side", "_rn")


def with_dup_of_pk(df_mail: DataFrame, df_existing: DataFrame) -> DataFrame:
    """mail 層去重：同 body_hash 取 create_date（再 pk_seq）最早者為正本，其餘填 dup_of_pk。
    候選池 = 表中既有列（排除本批 pk，因為本批會覆蓋它們）＋ 本批。body_hash NULL（短信 / 無正文）不參與。"""
    batch = df_mail.select("pk_seq", "body_hash", "create_date")
    pool = (
        df_existing.select("pk_seq", "body_hash", "create_date")
        .join(batch.select("pk_seq"), "pk_seq", "left_anti")
        .unionByName(batch)
        .where(F.col("body_hash").isNotNull())
    )
    w = Window.partitionBy("body_hash").orderBy("create_date", "pk_seq")
    first = pool.withColumn("_first_pk", F.first("pk_seq").over(w)).select("pk_seq", "_first_pk")
    return (
        df_mail.drop("dup_of_pk").join(first, "pk_seq", "left")
        .withColumn("dup_of_pk", F.when(F.col("_first_pk") != F.col("pk_seq"), F.col("_first_pk")))
        .select(*df_mail.columns)  # 保留原欄位順序
    )


# 隔離規則（D34，2026-10-23）：命中的信不寫 mail 表、記到 s_meeting_record_skipped_{unit}，其餘的信照常寫，流程不停。
# 之後有新規則往下加 (reason, SQL 條件)；reason 寫進隔離表 skip_reason，前面的規則優先。條件用字串、不用 F.col：
# 這個 cell 在 tests/ 沒有 SparkSession 時也要載得進。
SKIP_RULES = [
    ("from_unparsed", "forwarder_emp_id IS NULL"),  # mail_from 不符「工號/姓名/帳號/部門代碼」（測試信、外部寄件人）
]


def _shape(col):
    """字串形狀：數字→9、英文→A、中文→中；其餘（/ 空白 符號）保留。只看格式、不露姓名。"""
    return F.regexp_replace(F.regexp_replace(F.regexp_replace(col, r"\d", "9"), "[A-Za-z]", "A"), r"\p{IsHan}", "中")


def split_skipped(df_mail: DataFrame, skip_schema: StructType, cleaner_version: str) -> tuple[DataFrame, DataFrame]:
    """mail 表形狀的本批 → (可寫入的信, 隔離列)。隔離列欄位照 skip_schema；skip_detail 只放形狀與長度。
    parse_mail_from 不符格式時把原字串放 forwarder_name_zh（emp_id NULL），形狀就從那欄算。"""
    reason = F.lit(None).cast("string")
    for name, cond in reversed(SKIP_RULES):
        reason = F.when(F.expr(cond), F.lit(name)).otherwise(reason)
    tagged = df_mail.withColumn("_skip_reason", reason)
    raw_from = F.col("forwarder_name_zh")
    detail = F.when(
        F.col("_skip_reason") == "from_unparsed",
        F.concat(F.lit("mail_from="), F.coalesce(_shape(raw_from), F.lit("<null>")),
                 F.lit(" len="), F.coalesce(F.length(raw_from), F.lit(0)).cast("string")),
    )
    skipped = (
        tagged.where(F.col("_skip_reason").isNotNull())
        .select("pk_seq", "unit", "create_date", "mail_date", F.col("_skip_reason").alias("skip_reason"),
                detail.alias("skip_detail"), F.lit(cleaner_version).alias("cleaner_version"),
                F.current_timestamp().alias("skipped_at"), F.lit(None).cast("string").alias("note"))
        .select(*skip_schema.fieldNames())
    )
    kept = tagged.where(F.col("_skip_reason").isNull()).drop("_skip_reason")
    return kept, skipped

In [ ]:
# [c10] load_source
# T1-a：只取需要的欄、排除 is_test；增量以 mail 表 ∪ 隔離表的 max(create_date) 為水位（D14：不另建 job_run_log；
# D34：被跳過的信也推水位，不會每週再被撈），加上 cleaner_version 不同的舊列（重洗）、reprocess_pks 指定的 pk。
# backfill 或首跑用 mail_date 窗。
def fq(name: str) -> str:
    return f"{cfg['catalog']}.{cfg['schema']}.{name}"


fq_src = fq(cfg["source_table"])
fq_mail, fq_part = fq(f"s_meeting_record_mail_{cfg['unit']}"), fq(f"s_meeting_participants_{cfg['unit']}")
fq_skip = fq(f"s_meeting_record_skipped_{cfg['unit']}")  # D34 隔離表
fq_alias, fq_entity = fq("s_semantic_entity_alias"), fq("s_semantic_entity")
for t in (fq_mail, fq_part, fq_skip, fq_alias, fq_entity):
    assert spark.catalog.tableExists(t), f"{t} 不存在：先跑 migration 001 / 002（隔離表 2026-10-23 加，002 重跑一次即建）"

mail_tbl = spark.table(fq_mail)
src_all = spark.table(fq_src).select(*SOURCE_COLS).where(~F.coalesce(F.col("is_test"), F.lit(False)))
in_window = F.col("mail_date").between(F.lit(cfg["date_from"]), F.lit(cfg["date_to"]))

watermark = None
if cfg["run_mode"] == "backfill":
    df_src = src_all.where(in_window)
    scope = f"backfill mail_date {cfg['date_from']} ~ {cfg['date_to']}"
else:
    watermark = (mail_tbl.select("create_date").unionByName(spark.table(fq_skip).select("create_date"))
                 .agg(F.max("create_date")).collect()[0][0])
    if watermark is None:
        df_src = src_all.where(in_window)
        scope = f"incremental 首跑（表空）→ mail_date {cfg['date_from']} ~ {cfg['date_to']}"
    else:
        reclean = mail_tbl.where(F.col("cleaner_version") != cfg["cleaner_version"]).select("pk_seq")
        df_src = (
            src_all.where(F.col("create_date") > F.lit(watermark))
            .unionByName(src_all.join(reclean, "pk_seq", "left_semi"))
            .dropDuplicates(["pk_seq"])
        )
        scope = f"incremental create_date > {watermark}（＋ cleaner_version != {cfg['cleaner_version']} 重洗）"
if cfg["reprocess_pks"]:  # 隔離表的信處理好後（或任何要重洗的 pk）強制納入本批；is_test 仍排除
    df_src = df_src.unionByName(src_all.where(F.col("pk_seq").isin(cfg["reprocess_pks"]))).dropDuplicates(["pk_seq"])
    scope += f"；reprocess_pks {cfg['reprocess_pks']}"
if cfg["sample_limit"] > 0:
    df_src = df_src.orderBy(F.desc("create_date")).limit(cfg["sample_limit"])
    scope += f"；sample_limit {cfg['sample_limit']}"
print(scope)

In [ ]:
# [c11] run_transforms
# T1-m → 隔離（D34）→ mail 層去重 → T1-c → side / company。不 cache（serverless）：後面每個 action 會重算 UDF，日量百封級可接受。
mail_schema = spark.table(fq_mail).schema
skip_schema = spark.table(fq_skip).schema
part_schema = spark.table(fq_part).schema

# 先隔離再去重：被跳過的信不進 dup_of_pk 候選池，mail 表不會指到一封不存在的正本。
df_mail, df_skip = split_skipped(
    transform_mail(df_src, mail_schema, cfg["unit"], cfg["cleaner_version"]), skip_schema, cfg["cleaner_version"]
)
df_mail = (
    with_dup_of_pk(df_mail, mail_tbl)
    .withColumn("cleaned_at", F.current_timestamp())
    .select(*mail_schema.fieldNames())
)
df_part = (
    resolve_side(transform_participants(df_mail, part_schema), spark.table(fq_alias), spark.table(fq_entity))
    .withColumn("resolved_at", F.current_timestamp())
    .select(*part_schema.fieldNames())
)
df_mail.createOrReplaceTempView("_w1_mail_batch")
df_part.createOrReplaceTempView("_w1_part_batch")
df_skip.createOrReplaceTempView("_w1_skip_batch")
print("transforms defined (lazy)")

In [ ]:
# [c12] gates
# 關卡（規格 §4.0 / §4.0b / §4.1b）改成不停（D34，2026-10-23）：不符規則的信已在 [c11] 隔離、其餘照寫，這裡只印 WARN；
# 0 封是週跑的正常狀況。會停的只剩 [c30] 寫入後驗證。
# log：md 比例、來源分布、has_headers、display_len P50、oversize、短正文比例（D28）、整封重複、side unknown、對不到的網域。
def pct(n, d) -> float | None:
    return round(n / d, 3) if d else None


m = df_mail.agg(
    F.count("*").alias("n"),
    F.sum((F.col("body_source_field") == "mail_body_md").cast("int")).alias("n_md"),
    F.sum(F.col("body_conflict").cast("int")).alias("n_conflict"),
    F.sum(F.col("has_headers").cast("int")).alias("n_headers"),
    F.sum(F.col("oversize").cast("int")).alias("n_oversize"),
    F.sum((F.col("display_len") < MIN_BODY_CHARS).cast("int")).alias("n_short"),  # D28：初洗後不到 150 字（三欄都沒正文）
    F.sum(F.col("dup_of_pk").isNotNull().cast("int")).alias("n_dup"),
    F.sum(F.col("forwarder_emp_id").isNotNull().cast("int")).alias("n_from_ok"),
    F.sum(F.col("meeting_when").isNotNull().cast("int")).alias("n_when"),
    F.percentile_approx("display_len", 0.5).alias("display_len_p50"),
    F.percentile_approx("display_len", 0.95).alias("display_len_p95"),
    F.percentile_approx(F.col("display_len") / F.greatest(F.col("raw_len"), F.lit(1)), 0.5).alias("compress_p50"),
).collect()[0].asDict()
n = m["n"]
skip_by_reason = {r["skip_reason"]: r["c"] for r in
                  df_skip.groupBy("skip_reason").agg(F.count("*").alias("c")).collect()}
n_skip = sum(skip_by_reason.values())
skipped_pks = [r["pk_seq"] for r in df_skip.select("pk_seq").orderBy("pk_seq").limit(20).collect()]
source_dist = {r["body_source_field"]: r["c"] for r in
               df_mail.groupBy("body_source_field").agg(F.count("*").alias("c")).collect()}
p = df_part.agg(
    F.count("*").alias("rows"),
    F.countDistinct("pk_seq", "token").alias("tokens"),
    F.sum(((F.col("role") != "forwarder") & F.col("email").isNotNull()).cast("int")).alias("n_ext_email"),
    F.sum(((F.col("role") != "forwarder") & F.col("email").isNotNull() & (F.col("side") == "unknown")).cast("int")
          ).alias("n_unknown"),
    F.sum((F.col("kind") == "group").cast("int")).alias("n_group"),
).collect()[0].asDict()
unmatched = [(r["email_domain"], r["c"]) for r in
             df_part.where((F.col("side") == "unknown") & F.col("email_domain").isNotNull())
             .groupBy("email_domain").agg(F.count("*").alias("c")).orderBy(F.desc("c")).limit(20).collect()]

log = {
    "scope": scope, "mails_in_batch": n + n_skip, "mails": n, "mails_skipped": skip_by_reason,
    "skipped_pks_top20": skipped_pks, "md_rate": pct(m["n_md"], n), "source_dist": source_dist,
    "conflict_rate": pct(m["n_conflict"], n), "has_headers_rate": pct(m["n_headers"], n),
    "from_ok_rate": pct(m["n_from_ok"], n), "when_parsed": m["n_when"], "oversize": m["n_oversize"],
    "short_body_rate": pct(m["n_short"], n),
    "dup_of_pk_rate": pct(m["n_dup"], n), "display_len_p50": m["display_len_p50"],
    "display_len_p95": m["display_len_p95"], "compress_p50": m["compress_p50"],
    "participants": p, "side_unknown_rate": pct(p["n_unknown"], p["n_ext_email"]),
    "unmatched_domains_top20": unmatched,
}
print(json.dumps(log, ensure_ascii=False, default=str, indent=1))
if n + n_skip == 0:
    print("本批 0 封：水位之後沒有新信（週跑正常；後面 cell 空跑）")
if n_skip:
    print(f"WARN 隔離 {n_skip} 封 {skip_by_reason}，pk {skipped_pks}：不寫 mail 表、記到隔離表。"
          "測試信 → 請爬蟲標 is_test（或在隔離表 note 註記）；真信 → 調 [c04] MAIL_FROM_RE 後 reprocess_pks 重跑")
if (log["conflict_rate"] or 0) > 0.10:
    print("WARN body_conflict > 10%：三個正文欄不一致，回頭查 ETL（規格 §4.0b；信仍寫入，body_conflict 欄有記）")
if log["short_body_rate"] and log["short_body_rate"] > 0.10:
    print("WARN 初洗後 < 150 字的信 > 10%：三個正文欄都沒正文，查 ETL 或 [c03] 規則（D28：0.1 時 32% 是來源選錯）")
if log["side_unknown_rate"] and log["side_unknown_rate"] > 0.20:
    print("WARN side unknown > 20%：把上面 unmatched_domains 補進 001 [c05] 的 email_domain 別名")
if cfg["dry_run"]:
    print("\ndry_run = true：不寫表。樣本（不含姓名 / email / 正文）：")
    df_mail.select("pk_seq", "mail_date", "subject_clean", "forwarder_dept_code", "body_source_field",
                   "has_headers", "display_len", "dup_of_pk", "meeting_when").limit(10).show(truncate=60)
    df_part.select("pk_seq", "segment_no", "token", "role", "kind", "email_domain", "side", "company"
                   ).limit(20).show()
    df_skip.select("pk_seq", "mail_date", "skip_reason", "skip_detail").limit(10).show(truncate=80)

In [ ]:
# [c20] write_mail
# MERGE on pk_seq：同 pk 重洗就整列覆蓋（cleaner_version 升版、backfill 重跑都安全）。
# 隔離表（D34）：MERGE 不蓋人工 note；本批寫進 mail 表的 pk 從隔離表移除（= 處理完成）。
if not cfg["dry_run"]:
    spark.sql(f"""
        MERGE INTO {fq_mail} t USING _w1_mail_batch s ON t.pk_seq = s.pk_seq
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print("merged", fq_mail)
    spark.sql(f"""
        MERGE INTO {fq_skip} t USING _w1_skip_batch s ON t.pk_seq = s.pk_seq
        WHEN MATCHED THEN UPDATE SET unit = s.unit, create_date = s.create_date, mail_date = s.mail_date,
            skip_reason = s.skip_reason, skip_detail = s.skip_detail, cleaner_version = s.cleaner_version,
            skipped_at = s.skipped_at
        WHEN NOT MATCHED THEN INSERT *
    """)
    spark.sql(f"DELETE FROM {fq_skip} WHERE pk_seq IN (SELECT pk_seq FROM _w1_mail_batch)")
    print("merged", fq_skip, "| removed pks now written to mail table")

In [ ]:
# [c21] write_participants
# 本批 pk 的參與者整組重建：先 DELETE 該 pk 的舊列（重解析後 token 可能變），再 append。
# 不用 MERGE：鍵 (pk_seq, segment_no, token, role) 在重解析時可能整組換掉，MERGE 留不住「消失的列」的刪除。
if not cfg["dry_run"]:
    spark.sql(f"DELETE FROM {fq_part} WHERE pk_seq IN (SELECT pk_seq FROM _w1_mail_batch)")
    df_part.write.mode("append").saveAsTable(fq_part)
    print("rebuilt participants for batch →", fq_part)

In [ ]:
# [c30] check_target
# 寫入後驗證：本批 pk 都在 mail 表且 cleaner_version 正確；參與者鍵無重複；隔離列都在、沒有 pk 同時在兩張表（D34）；
# clean_text 硬規則在 T1-b 檢。
if not cfg["dry_run"]:
    batch_pks = spark.table("_w1_mail_batch").select("pk_seq")
    mail_now = spark.table(fq_mail).join(batch_pks, "pk_seq", "left_semi")
    part_now = spark.table(fq_part).join(batch_pks, "pk_seq", "left_semi")
    stale = mail_now.where(F.col("cleaner_version") != cfg["cleaner_version"]).count()
    dup_keys = (part_now.groupBy("pk_seq", "segment_no", "token", "role").count().where("count > 1").count())
    no_part = mail_now.where("has_headers").join(
        part_now.where("role <> 'forwarder'").select("pk_seq").distinct(), "pk_seq", "left_anti").count()
    skip_now = spark.table(fq_skip)
    both = skip_now.join(spark.table(fq_mail).select("pk_seq"), "pk_seq", "left_semi").count()
    skip_in_table = skip_now.join(spark.table("_w1_skip_batch").select("pk_seq"), "pk_seq", "left_semi").count()
    print({"batch_mails_in_table": mail_now.count(), "participant_rows": part_now.count(),
           "batch_skipped_in_table": skip_in_table, "skipped_total": skip_now.count(), "pk_in_both_tables": both,
           "stale_cleaner_version": stale, "dup_participant_keys": dup_keys,
           "has_headers_but_no_participants": no_part,
           "table_total": spark.table(fq_mail).count()})
    assert stale == 0, "有本批 pk 的 cleaner_version 不是本次版本"
    assert dup_keys == 0, "參與者鍵重複"
    assert both == 0, "同一 pk 同時在 mail 表與隔離表（[c20] 的 DELETE 沒做完）"
    if no_part:
        print("WARN 有標頭但沒解析出參與者的信：", [r.pk_seq for r in
              mail_now.where("has_headers").join(part_now.where("role <> 'forwarder'").select("pk_seq").distinct(),
                                                 "pk_seq", "left_anti").select("pk_seq").limit(20).collect()])